# First AUP-ZU3 waveform comparison

On the board, run these cells in order. The first cell identifies the build. The second programs the fabric. See [the setup instructions](README.md) before starting.

In [ ]:
import hashlib
import platform
import sys
from pathlib import Path

import pynq

repo = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "hardware/pynq/host.py").is_file())
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "hardware/pynq"))

from host import IP_NAME, load_overlay
from software.golden_model import preprocess, quantize

bitfile = repo / "hardware/build/overlay/pocketfilter.bit"
print("System", platform.platform(), "Python", platform.python_version(), "PYNQ", pynq.__version__)
for artifact in (bitfile, bitfile.with_suffix(".hwh")):
    print(artifact.name, "SHA256", hashlib.sha256(artifact.read_bytes()).hexdigest())

In [ ]:
overlay, kernel = load_overlay(bitfile)
for name, metadata in overlay.ip_dict[IP_NAME]["registers"].items():
    print(name, hex(metadata["address_offset"]), metadata["size"], "bits")

Each row prints the four FPGA words after exact comparison with the Python golden model. Indices are zero-based and inclusive. The first case runs the team's equal-waveform vector through median width 1, normalization and quantization. The second uses signed integer codes with a nonzero distance. The third has no feasible width-zero path.

In [ ]:
identical = quantize(preprocess([0, 1, 0], kernel_size=1))
cases = [
    ("identical", identical, identical, 0),
    ("signed offset", [40, -5, 12, 30, 45], [-4, 13], 0),
    ("no path", [7], [7, 7], 0),
]
for name, signal, reference, width in cases:
    words = kernel.compare(signal, reference, width)
    print(name, words)